# STARRT: fourth GPU round

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. *Add Input* → the `starrt-gpu4` dataset.

Run cell 1 and the smoke test first. Then **Save Version → Save & Run All (Commit)**. Total about 6-7 hours. Results → `starrt_gpu4_results.zip`.

In [ ]:
import glob, os, subprocess
hits = glob.glob('/kaggle/input/**/run_distilbert.py', recursive=True)
assert hits, 'Dataset not attached: Add Input -> starrt-gpu4'
CODE = os.path.dirname(hits[0]); print('code folder:', CODE)
import torch; print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run(script, args=''):
    cmd = f'cd /kaggle/working && python -u {CODE}/{script} {args}'
    print('>>', cmd, flush=True); r = subprocess.run(cmd, shell=True)
    if r.returncode != 0: print('!! FAILED:', script, args, '(continuing with the next step)')

## 0. Smoke test (3 min)

In [ ]:
run('run_distilbert.py', '--datasets djinni_tools --folds 0 --epochs 1 --limit 200 --skip_forms --tag smoke')
run('run_llm_zeroshot.py', '--datasets faruque --folds 1 --limit 2 --shots 1 --per_class 20 --tag smoke_llm')
print('smoke test done: check above that both printed a [smoke ...] result line')

## A. Few-shot Qwen2.5-7B: one labelled example per field from the training folds, 20 rows per class from each of the five test folds (~3.5 h)

In [ ]:
run('run_llm_zeroshot.py', '--model Qwen/Qwen2.5-7B-Instruct --device_map auto --datasets djinni,faruque --folds 0,1,2,3,4 --per_class 20 --shots 1 --tag llm_qwen2.5-7b_1shot')

## B. DistilBERT on the Djinni ablations: without 'looking for', role words masked, tool list only (~1.5 h)

In [ ]:
run('run_distilbert.py', '--datasets djinni_nolook,djinni_masked,djinni_tools --skip_forms --tag distilbert_ablation')

## C. (if time is left) Few-shot Qwen2.5-3B, same rows (~1.2 h)

In [ ]:
run('run_llm_zeroshot.py', '--datasets djinni,faruque --folds 0,1,2,3,4 --per_class 20 --shots 1 --tag llm_qwen2.5-3b_1shot')

## Zip results
Download `starrt_gpu4_results.zip` from the **Output** tab and send it in the chat.

In [ ]:
import shutil
for f in glob.glob('/kaggle/working/results/smoke*'): os.remove(f)
shutil.make_archive('/kaggle/working/starrt_gpu4_results', 'zip', '/kaggle/working/results')
print(sorted(os.listdir('/kaggle/working/results')))